# Case Study: Cyclistic Bike-Share - Converting Casual Riders to Members

<br>

**Author:** Jacek Sikorski  
**Project:** Google Data Analytics Capstone  
**Tools:** Google BigQuery SQL, Google Sheets, Python pandas, Jupyter Notebook  
**Date:** 2026

<br>

Cyclistic is a bike-share company operating in Chicago. The company offers both casual rides and annual memberships. While casual riders generate significant usage, annual members provide more stable and predictable revenue.

The objective of this case study is to analyze historical trip data to identify behavioral differences between casual riders and members, and provide data-driven recommendations to support membership conversion strategy.

<div id="ask" style="border-bottom: 2px solid #444; padding-bottom: 6px; margin-top: 50px; margin-bottom: 10px;">
<h2>⚫ 1. Ask</h2>
</div>

### Business Task

To analyze behavioral differences between casual riders and annual members in order to support data-driven membership conversion strategies. The insights will inform targeted marketing initiatives aimed at increasing annual membership adoption.


### Key Stakeholders

* Lily Moreno, Director of Marketing

* Cyclistic Executive Team

### Primary Question

How do casual and member riders differ in their usage patterns?


### Analytical Focus

The analysis will examine ride duration, ride frequency, and temporal usage patterns.

<div id="prepare" style="border-bottom: 2px solid #444; padding-bottom: 6px; margin-top: 50px; margin-bottom: 10px;">
<h2>⚫ 2. Prepare</h2>
</div>

### Data Source


Cyclistic historical trip data (12 months), collected from: [divvy-tripdata](https://divvy-tripdata.s3.amazonaws.com/index.html). The dataset is publicly available and provided for analytical purposes by Motivate International Inc. While it represents real operational data, it may contain recording inconsistencies typical of transactional systems. No direct personal identifiers are included.
###### _[The data has been made available by Motivate International Inc. under this [license](https://divvybikes.com/data-license-agreement).]_

<br>

### Data Collection


Twelve monthly CSV files were downloaded and uploaded to a Google Cloud Storage bucket. Corresponding tables were created in the 'cyclistic_case_study' dataset in BigQuery. The dataset covers the period from __February 2025 to January 2026__ (table names schema: '2025-02' - '2026-01'). The dataset contains ride timestamps, membership classification, and ride characteristics, which are sufficient to compare behavioral patterns between casual and member riders.

<br>


### Tools Used for The Case

* Data cleaning and processing - Google Cloud BigQuery, Python pandas

* Data visualisation - Google Sheets

* Documentation - Jupyter Notebook

<br>



<div style="
background-color:#d1d1d1;
padding:14px;
border-left:4px solid #6e6868;
margin-top:10px;
margin-bottom:15px;
">

Initial inspection identified missing values in certain fields, which are handled in the data processing stage. However, inspection showed consistent schema across all 12 monthly files. Each table includes following 13 variables, with data types as below: 

</div>

In [2]:
import pandas as pd
from google.cloud import bigquery

client = bigquery.Client()

data_types_query = """ 
SELECT 
  column_name, data_type 
FROM 
  cyclistic_case_study.INFORMATION_SCHEMA.COLUMNS 
WHERE 
  table_name = '2026-01'
""" 
df_data_types = client.query(data_types_query).to_dataframe() 
df_data_types

,column_name,data_type
0,ride_id,STRING
1,rideable_type,STRING
2,started_at,TIMESTAMP
3,ended_at,TIMESTAMP
4,start_station_name,STRING
5,start_station_id,STRING
6,end_station_name,STRING
7,end_station_id,STRING
8,start_lat,FLOAT64
9,start_lng,FLOAT64


<div id="process" style="border-bottom: 2px solid #444; padding-bottom: 6px; margin-top: 50px; margin-bottom: 10px;">
<h2>⚫ 3. Process</h2>
</div>

### 3.1 Analytical Scope Definition and Initial Data Exploration

<br>

Since the analytical scope focuses on ride duration and temporal patterns, station-level attributes were evaluated but not required for further analysis. Following columns were included to the process:

* ride_id

* started_at

* ended_at

* member_casual

<div style="
background-color:#d1d1d1;
padding:14px;
border-left:4px solid #6e6868;
margin-top:10px;
margin-bottom:15px;
">

The dataset contains two clearly defined rider categories: members and casual riders. Both groups have sufficient representation for comparative behavioral analysis:

</div>


In [3]:
member_casual_query = """ 
SELECT 
  member_casual, 
  COUNT(*) AS ride_count 
FROM 
  `cyclistic_case_study.202*` -- The wildcard '202*' allows the query to read data 
GROUP BY                      -- from multiple tables whose names start with 202,
  member_casual;              -- combining all monthly tables into a single query.
""" 
df_member_casual = client.query(member_casual_query).to_dataframe() 
df_member_casual

,member_casual,ride_count
0,casual,2000093
1,member,3551999


### 3.2 Null Value Assessment

<br>

<div style="
background-color:#d1d1d1;
padding:14px;
border-left:4px solid #6e6868;
margin-top:10px;
margin-bottom:10px;
">

Missing values are primarily present in station-related columns. As the analysis does not require station-level geospatial insights, these fields were retained but excluded from decision criteria:

</div>

In [4]:
nulls_query = """
SELECT 
  COUNTIF(ride_id IS NULL) AS ride_id_nulls,
  COUNTIF(rideable_type IS NULL) AS rideable_type_nulls,
  COUNTIF(started_at IS NULL) AS started_at_nulls,
  COUNTIF(ended_at IS NULL) AS ended_at_nulls,
  COUNTIF(start_station_name IS NULL) AS start_station_name_nulls,
  COUNTIF(start_station_id IS NULL) AS start_station_id_nulls,
  COUNTIF(end_station_name IS NULL) AS end_station_name_nulls,
  COUNTIF(end_station_id IS NULL) AS end_station_id_nulls,
  COUNTIF(start_lat IS NULL) AS start_lat_nulls,
  COUNTIF(start_lng IS NULL) AS start_lng_nulls,
  COUNTIF(end_lat IS NULL) AS end_lat_nulls,
  COUNTIF(end_lng IS NULL) AS end_lng_nulls,
  COUNTIF(member_casual IS NULL) AS member_casual_nulls
FROM 
  `cyclistic_case_study.202*`
"""
df_nulls = client.query(nulls_query).to_dataframe() 
df_nulls

,ride_id_nulls,rideable_type_nulls,started_at_nulls,ended_at_nulls,start_station_name_nulls,start_station_id_nulls,end_station_name_nulls,end_station_id_nulls,start_lat_nulls,start_lng_nulls,end_lat_nulls,end_lng_nulls,member_casual_nulls
0,0,0,0,0,1186374,1186374,1246827,1246827,0,0,5670,5670,0


<br>

### 3.3 Duplicate Validation

<br>

<div style="
background-color:#d1d1d1;
padding:14px;
border-left:4px solid #6e6868;
margin-top:10px;
margin-bottom:15px;
">

No duplicate ride_id records were detected:

</div>

In [5]:
minmax_query = """ 
SELECT 
  MIN(TIMESTAMP_DIFF(ended_at, started_at, MINUTE)) AS min_duration, 
  MAX(TIMESTAMP_DIFF(ended_at, started_at, MINUTE)) AS max_duration 
FROM 
  `cyclistic_case_study.202*` 
""" 
df_minmax = client.query(minmax_query).to_dataframe() 
df_minmax

,min_duration,max_duration
0,-54,1574


### 3.4 Duration Validation

<br>

<div style="
background-color:#d1d1d1;
padding:14px;
border-left:4px solid #6e6868;
margin-top:10px;
margin-bottom:15px;
">

Duration validation revealed the presence of negative ride lengths, which were removed as invalid records. Additionally, rides exceeding 24 hours were considered anomalous and excluded to ensure analytical consistency in subsequent analysis:

</div>

In [ ]:
SELECT
  MIN(TIMESTAMP_DIFF(ended_at, started_at, MINUTE)) AS min_duration,
  MAX(TIMESTAMP_DIFF(ended_at, started_at, MINUTE)) AS max_duration
FROM `cyclistic_case_study.202*`

### 3.5 Cleaned Analytical View

<br>

<div style="
background-color:#d1d1d1;
padding:14px;
border-left:4px solid #6e6868;
margin-top:10px;
margin-bottom:15px;
">

A dedicated analytical view was created to centralize data cleaning logic and derived features. This approach ensures consistency across all subsequent queries while avoiding duplication of transformation logic.
Monthly, day-of-week and hourly features were extracted to support temporal usage analysis and segment comparison. Ride start time was selected as the temporal reference point to reflect rider behavior at trip initiation:

</div>

In [ ]:
CREATE OR REPLACE VIEW `cyclistic_case_study.cleaned_view` AS
WITH base AS 
  (
  SELECT
    *,
    TIMESTAMP_DIFF(ended_at, started_at, MINUTE) AS ride_length
  FROM `cyclistic_case_study.202*`
  )
SELECT
  *,
  EXTRACT(MONTH FROM started_at) AS ride_month_number,
  EXTRACT(DAYOFWEEK FROM started_at) AS day_of_week,
  EXTRACT(HOUR FROM started_at) AS hour_of_day
FROM base
WHERE 
  ride_length > 0 AND ride_length <= 1440;

<div id="analyze" style="border-bottom: 2px solid #444; padding-bottom: 6px; margin-top: 50px; margin-bottom: 10px;">
<h2>⚫ 4. Analyze</h2>
</div>

### 4.1 Ride Distribution and Duration Comparison

<br>

__Objective__

The objective of this analysis is to compare overall ride volume and ride duration patterns between annual members and casual riders in order to identify structural behavioral differences.



The following SQL query was used to generate the aggregated data for the two subsequent visualizations:

* Figure 1. Total Rides by Membership Type

* Figure 2. Average and Median Ride Duration by Membership Type

In [ ]:
SELECT
  member_casual,
  COUNT(*) AS total_rides,
  AVG(ride_length) AS avg_ride_length,
  APPROX_QUANTILES(ride_length, 100)[OFFSET(50)] AS median_ride_length
FROM `cyclistic_case_study.cleaned_view`
GROUP BY member_casual;

<br>

<p align="center">
<img src="https://raw.githubusercontent.com/sicztery/cyclistic_bike_share_case_study/refs/heads/main/images/fig1.png" width="700">
</p>

<p style="text-align:center; font-style:italic; font-size:10px;">
Figure 1. Total Rides by Membership Type
</p>

Members account for approximately 65% of total rides, while casual riders represent 35%. This indicates that annual members generate the majority of overall platform usage, suggesting higher engagement frequency compared to casual riders (Fig. 1).

<br>


<p align="center">
<img src="https://raw.githubusercontent.com/sicztery/cyclistic_bike_share_case_study/refs/heads/main/images/fig2.png" width="700">
</p>

<p style="text-align:center; font-style:italic; font-size:10px;">
Figure 2. Average and Median Ride Duration by Membership Type
</p>

The average ride duration for casual riders is noticeably higher than for members. However, the median values are much closer, suggesting that longer casual rides have a disproportionate impact on the average (Fig. 2).

<br>


### 4.2 Weekly Usage Patterns: Weekday vs Weekend

<br>

__Objective__

To determine whether casual and member riders differ in their weekly usage patterns, particularly between weekdays and weekends.



The following SQL query was used to generate the aggregated data for the two subsequent visualizations:

* Figure 3. Ride Volume by Membership Type: Weekday vs Weekend

* Figure 4. Average and Median Ride Duration by Membership Type and Day Category

In [ ]:
SELECT
  member_casual,
  CASE 
    WHEN day_of_week IN (1,7) THEN 'Weekend'
    ELSE 'Weekday'
  END AS day_type,
  COUNT(*) AS total_rides,
  AVG(ride_length) AS avg_ride_length,
  APPROX_QUANTILES(ride_length, 100)[OFFSET(50)] AS median_ride_length
FROM `cyclistic_case_study.cleaned_view`
GROUP BY member_casual, day_type
ORDER BY member_casual, day_type;

<br>

<p align="center">
<img src="https://raw.githubusercontent.com/sicztery/cyclistic_bike_share_case_study/refs/heads/main/images/fig3.png" width="700">
</p>

<p style="text-align:center; font-style:italic; font-size:10px;">
Figure 3. Ride Volume by Membership Type: Weekday vs Weekend
</p>

Members generate the majority of weekday rides, indicating routine or commuting-oriented usage. In contrast, casual riders account for a relatively strong share of weekend activity despite representing a smaller overall segment (Fig. 3).

<br>


<p align="center">
<img src="https://raw.githubusercontent.com/sicztery/cyclistic_bike_share_case_study/refs/heads/main/images/fig4.png" width="700">
</p>

<p style="text-align:center; font-style:italic; font-size:10px;">
Figure 4. Average and Median Ride Duration by Membership Type and Day Category
</p>

Foregoing chart shows consistent differences in ride duration between membership types. Casual riders record longer average ride durations than members both on weekdays and weekends.

The gap between average and median values is more pronounced within the casual segment, indicating that a portion of longer rides increases the overall average. In contrast, member ride durations remain relatively more stable across both time categories.

Additionally, ride durations increase during weekends for both groups, with the longest rides observed among casual riders on weekends (Fig. 4).

<br>


### 4.3 Monthly Usage Patterns

<br>

__Objective__

To determine whether casual and member riders differ in their monthly usage patterns, especially during shoulder months.



The following SQL query was used to generate the aggregated data for monthly usage analysis (Fig. 5):

In [ ]:
WITH monthly AS (
  SELECT
    ride_month_number,
    FORMAT_DATE('%b', DATE(started_at)) AS ride_month_name,
    member_casual
  FROM `cyclistic_case_study.cleaned_view`
)
SELECT
  ride_month_number,
  ride_month_name,
  member_casual,
  COUNT(*) AS total_rides
FROM monthly
GROUP BY ride_month_number, ride_month_name, member_casual
ORDER BY ride_month_number;

<br>

<p align="center">
<img src="https://raw.githubusercontent.com/sicztery/cyclistic_bike_share_case_study/refs/heads/main/images/fig5.png" width="700">
</p>

<p style="text-align:center; font-style:italic; font-size:10px;">
Figure 5. Monthly Ride Volume by Membership Type
</p>

Both segments exhibit clear seasonal growth toward summer months. However, casual ridership declines more sharply after its peak, while member ridership remains relatively stable through early autumn. This pattern suggests stronger seasonal dependence within the casual segment (Fig. 5).

<br>
<br>


### 4.4 Day-of-Week and Hourly Usage Pattern

<br>

__Objective__

To analyze how ride frequency and ride duration vary across individual hours and days of the week for each membership segment.



The two following SQL queries were used to generate the aggregated data for subsequent visualizations:

* Figure 6. Average Ride Duration by Day of Week

* Figure 7. Hourly Ride Volume by Membership Type

In [ ]:
SELECT
  MOD(day_of_week + 5, 7) + 1 AS weekday_order,    -- Shifts BigQuery's DAYOFWEEK (Sunday-first)
  FORMAT_DATE('%A', DATE(started_at)) AS day_name, -- to a Monday-first order
  member_casual,
  COUNT(*) AS total_rides,
  AVG(ride_length) AS avg_ride_duration
FROM `cyclistic_case_study.cleaned_view`
GROUP BY weekday_order, day_name, member_casual
ORDER BY weekday_order, member_casual;

In [ ]:
SELECT
  hour_of_day,
  member_casual,
  COUNT(*) AS total_rides
FROM `cyclistic_case_study.cleaned_view`
GROUP BY hour_of_day, member_casual
ORDER BY hour_of_day;

<br>

<p align="center">
<img src="https://raw.githubusercontent.com/sicztery/cyclistic_bike_share_case_study/refs/heads/main/images/fig6.png" width="700">
</p>

<p style="text-align:center; font-style:italic; font-size:10px;">
Figure 6. Average Ride Duration by Day of Week
</p>

Weekly patterns further differentiate the two rider segments. Member activity remains relatively stable throughout weekdays, indicating routine usage. In contrast, casual ridership peaks on Saturdays and Sundays, with noticeably longer ride durations. This reinforces the distinction between routine-based and leisure-oriented usage patterns (Fig. 6).

<br>


<p align="center">
<img src="https://raw.githubusercontent.com/sicztery/cyclistic_bike_share_case_study/refs/heads/main/images/fig7.png" width="700">
</p>

<p style="text-align:center; font-style:italic; font-size:10px;">
Figure 7. Hourly Ride Volume by Membership Type
</p>

Hourly analysis reveals distinct intra-day patterns between rider segments. Member activity peaks during morning and late afternoon hours, consistent with commuting behavior. In contrast, casual ridership is more evenly distributed throughout the day, with relatively stronger activity during midday and afternoon hours.

This further reinforces the distinction between routine-based and discretionary usage patterns across segments (Fig. 7).

<br>


### 4.5 Analysis Summary

<br>

The analysis reveals consistent behavioral differences between annual members and casual riders across multiple dimensions of platform usage.

Members generate the majority of rides and demonstrate relatively stable usage patterns across weekdays and throughout the year. Their activity peaks during typical commuting hours, indicating routine-based usage associated with daily mobility.

Casual riders, in contrast, exhibit longer ride durations and stronger weekend participation. Their usage also shows greater seasonal fluctuation and more evenly distributed activity throughout the day, suggesting leisure-oriented riding behavior.

These patterns highlight clear distinctions in how each segment interacts with the bike-sharing service and provide a foundation for targeted strategic recommendations.

<div id="share" style="border-bottom: 2px solid #444; padding-bottom: 6px; margin-top: 50px; margin-bottom: 10px;">
<h2>⚫ 5. Share</h2>
</div>

### Key Insights

<br>

__Objective__

To summarize the key behavioral differences between casual riders and annual members and present the most relevant findings supporting strategic decision-making.

<br>


__Insight 1__: Members generate the majority of total rides

Annual members account for the largest share of total rides on the platform. Their usage remains relatively stable throughout weekdays and across most months of the year. This pattern suggests that members rely on Cyclistic bikes as part of routine transportation rather than occasional use.

<br>


__Insight 2__: Casual riders take significantly longer trips

Casual riders exhibit longer ride durations on average compared to members. The difference between average and median ride duration also indicates that a portion of casual trips are substantially longer. This suggests that casual riders are more likely to use the service for leisure-oriented activities rather than short routine trips.

<br>

__Insight 3__: Members show strong commuting patterns

Hourly ride analysis reveals clear peaks in member activity during morning and late afternoon hours. This pattern is consistent with commuting behavior and indicates that many members integrate bike usage into daily travel routines.

<br>

__Insight 4__: Casual riders are more weekend-oriented

Casual riders show relatively stronger participation during weekends compared to members. Combined with longer ride durations, this indicates that casual usage is more closely associated with recreational riding rather than weekday transportation needs.

<br>

__Insight 5__: Seasonal patterns affect both rider segments differently

Both rider segments show increased activity during warmer months. However, casual ridership declines more sharply after the summer peak, while member ridership remains relatively stable through early autumn before also decreasing during the winter months. This suggests that casual riders are more immediately influenced by seasonal conditions, while members maintain more consistent usage before the broader seasonal decline affects overall ridership.

<br>


### Segment Behavior Overview

<br>

The analysis highlights two clearly differentiated rider profiles within the Cyclistic platform.

Annual members primarily use the service as part of regular urban mobility. Their activity is concentrated on weekdays and during typical commuting hours, indicating routine and transportation-oriented usage.

Casual riders display a different pattern of engagement. Their rides tend to be longer, occur more frequently during weekends, and show stronger seasonal fluctuations. These characteristics suggest that casual riders are more likely to use the service for discretionary or leisure-oriented trips rather than daily transportation.

Overall, the findings indicate that the two segments interact with the bike-sharing system in fundamentally different ways. Understanding these behavioral differences provides a foundation for strategies aimed at converting casual riders into annual members.

<div id="act" style="border-bottom: 2px solid #444; padding-bottom: 6px; margin-top: 50px; margin-bottom: 10px;">
<h2>⚫ 6. Act</h2>
</div>

### Recommendations

<br>

__Recommendation 1__: Target casual riders during peak seasonal usage

Casual ridership increases significantly during the summer months and declines rapidly afterward. This seasonal engagement window presents an opportunity to introduce membership offers when casual riders are most active and receptive to the service.

Cyclistic could intensify marketing campaigns during peak riding months, focusing on converting frequent casual users into annual members. Promotional messaging could emphasize cost savings for riders who regularly take longer leisure trips.

This strategy leverages the period when casual riders already demonstrate the highest engagement with the platform.

<br>

__Recommendation 2__: Promote membership after long casual rides

Casual riders tend to take significantly longer trips than members. These extended rides may indicate higher engagement and a greater probability that the rider will continue using the service in the future.

Cyclistic could implement targeted in-app or post-ride messaging encouraging membership after rides exceeding a defined duration or distance threshold. Highlighting potential savings and convenience could increase the likelihood of conversion among highly engaged casual riders.

This approach targets riders who have already demonstrated strong usage behavior.

<br>

__Recommendation 3__: Emphasize membership as a commuting solution

Members exhibit clear weekday commuting patterns with peak activity during morning and evening hours. This indicates that membership aligns well with routine transportation needs.

Marketing communication should highlight membership as a reliable and cost-effective option for daily mobility, particularly for urban commuters. Campaigns could focus on convenience, predictability of travel, and the benefits of integrating bike-sharing into daily transportation routines.

This messaging aligns the membership product with the usage patterns already demonstrated by existing members.

<br>

### Conclusion

<br>

This analysis demonstrates that Cyclistic’s two primary rider segments interact with the bike-sharing system in fundamentally different ways. Annual members integrate the service into routine urban mobility, while casual riders engage with the platform more opportunistically and recreationally.

Recognizing these distinct behavioral patterns creates an opportunity to design targeted conversion strategies that align with how casual riders already use the service. By focusing on high-engagement moments and seasonal peaks, Cyclistic can position membership as a natural progression from occasional usage to regular mobility.

Understanding rider behavior in this way provides a foundation for more effective marketing initiatives and long-term growth of the membership base.